# Empirical Risk Premia Engine v0.2.3 — Funded Rates adapter

Portable notebook with embedded v0.2 engine. Reuses the validated v0.8 **Level/Slope/Curvature funded factor** construction, emits common outputs, but deliberately does **not** reimplement the v0.8 Fama–Bliss/CP conditional trading grid. Source: original Federal Reserve Board Kim-Wright bulk CSV (10 zero yields, one request), with exact monthly FEDFUNDS funding. No Treasury par-yield substitute.

In [ ]:
# Offline-bundled erp v0.2.3 sources: this notebook also works when uploaded alone to Colab.
import io, zipfile, base64, sys, subprocess, importlib.util
from pathlib import Path
ROOT = Path.cwd() / "erp_v023_notebook_runtime"
(ROOT / "src").mkdir(parents=True, exist_ok=True)
PAYLOAD = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as zf:
    zf.extractall(ROOT / "src")
for module, package in {"numpy":"numpy","pandas":"pandas","scipy":"scipy","statsmodels":"statsmodels","requests":"requests","openpyxl":"openpyxl"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package])
sys.path.insert(0,str(ROOT / "src"))
# In a reused Colab runtime, purge previously imported older portable packages.
for _name in list(sys.modules):
    if _name == "erp" or _name.startswith("erp."):
        del sys.modules[_name]
import numpy as np
import pandas as pd
from IPython.display import display
from erp import __version__, export_snapshot, EquityData, EquityPremiaEngine, RatesData, RatesPremiaEngine
print("ERP version:",__version__, "| portable package embedded and loaded")


### Reliable source configuration

`CURVE_SOURCE="board"` uses one official Federal Reserve CSV rather than 10 slow FRED requests. If automated access fails, open `https://www.federalreserve.gov/data/yield-curve-tables/feds200533.csv` in your browser, upload the file to Colab and assign `BOARD_CSV_PATH`. Funding uses the **exact monthly FEDFUNDS** series. The funding provider is a separate publicly available CSV mirror of the original FRED/FEDFUNDS monthly series (not DBnomics), with optional recent-month Board H.15 update. The notebook will **warn and truncate its sample** if funding values do not extend to the selected `AS_OF`, rather than carry a stale rate forward. For a fully network-independent funding load, download either `https://eco3min.fr/dataset/federal-funds-rate.csv` or `https://fred.stlouisfed.org/graph/fredgraph.csv?id=FEDFUNDS` in the browser and upload the CSV; assign `FEDFUNDS_CSV_PATH`. Neither funding nor curve is silently substituted.

In [ ]:
AS_OF = "2026-10-31"
OUT = Path('erp_v02_output')
CACHE_DIR = Path('erp_data_cache')
CURVE_SOURCE = 'board'   # single official Fed Board CSV
BOARD_CSV_PATH = None    # e.g. '/content/feds200533.csv' (upload original Board CSV)
FUNDING_SOURCE = 'mirror' # 'mirror' (default), 'fred', 'auto', 'local'
FEDFUNDS_CSV_PATH = None # e.g. '/content/FEDFUNDS.csv' (local source takes precedence)


## 1 — Zero curve, funding proxy and standardized PCA

Ex-post factor extraction: lagged EWMA risk and static PCA calibration ending 2002-12-31; positive Level means upward yields. No ex-ante forecast is claimed.

In [ ]:
rates_data = RatesData.from_fred(
    cache_dir=CACHE_DIR, curve_source=CURVE_SOURCE,
    local_curve_path=BOARD_CSV_PATH,
    local_fedfunds_path=FEDFUNDS_CSV_PATH, funding_source=FUNDING_SOURCE,
)
rates = RatesPremiaEngine(rates_data)
snapshot = rates.fit(as_of=AS_OF)
print(snapshot.metadata)
display(snapshot.factor_premia.to_frame('mean_monthly'))
display(rates.pca.loadings)

## 2 — Current funded mimicking portfolios and risk decomposition

In [ ]:
print('Current risky weights and cash funding')
display(rates.latest_weights.round(4))
display(rates.latest_mimic_diagnostics.round(8))
print('Current one-sigma yield-shock ZC exposures')
display(rates.latest_analytic_exposures.round(5))
print('Carry and curve decomposition')
display(pd.concat({'FundedCarry':rates.factor_carry.mean(), 'RealizedCurve':rates.factor_curve_component.mean()},axis=1))

## 3 — Export standard core schema + rates-specific analytic modules

In [ ]:
destination=export_snapshot(snapshot,OUT/'rates_PCA3_FUNDED')
rates.export_auxiliary(destination)
print('Exported:', destination)
print('Files:',sorted(x.name for x in destination.iterdir()))